<a href="https://colab.research.google.com/github/NancyWangareh/ReCover/blob/main/Information_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ReCover: Microinsurance Advisor for Kenyan Informal Traders

A retrieval-augmented chatbot that answers insurance questions for market traders (e.g. mitumba and shoe vendors) in English, Kiswahili or Sheng.

| Requirement | Where it is handled |
|---|---|
| LLM interprets everyday-language questions | `interpret()`: Gemini turns a message into structured fields (trade, structure, stock value, budget, intent, language) |
| Retrieve from approved documents | `retrieve()`: ChromaDB + `all-MiniLM-L6-v2`, with a minimum-similarity cutoff |
| Rules for eligibility, coverage, premiums, exclusions | `evaluate()`: plain Python rules over structured product data, no LLM involved |
| Source references | Every excerpt and rule result gets a label (`[S1]`, `[R1]`); the answer is returned with the matching source list |
| Say when information is unavailable | Hard guard in `ReCover.ask()` (no evidence = no LLM call) plus missing fields reported as "not in approved documents" |

**Run locally:** `pip install -r requirements.txt`, set `GEMINI_API_KEY`, open the notebook from the repo root.
**Run in Colab:** add `GEMINI_API_KEY` under Secrets.

1. Cloning the repository and inspecting the data we have before building the model


In [16]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/NancyWangareh/ReCover.git"

REPO_ROOT = Path("/content/ReCover")
DOCS_DIR = REPO_ROOT / "data collection"

if not REPO_ROOT.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_ROOT)],
        check=True
    )
else:
    print("ReCover repository already exists.")

if not DOCS_DIR.exists():
    raise FileNotFoundError(
        f"Documents folder not found: {DOCS_DIR}"
    )

print("Repository:", REPO_ROOT)
print("Documents:", DOCS_DIR)

ReCover repository already exists.
Repository: /content/ReCover
Documents: /content/ReCover/data collection


In [17]:
%pip install -q pypdf

from pathlib import Path
from pypdf import PdfReader

for pdf in sorted(Path("ReCover/data collection").glob("*.pdf")):
    reader = PdfReader(str(pdf))
    chars = sum(len(page.extract_text() or "") for page in reader.pages)
    print(f"{pdf.name}: {len(reader.pages)} pages, {chars:,} characters of text")

BRITAM-PRODUCTS-PROFILE-Final-Final.pdf: 20 pages, 0 characters of text
TCIP-BROCHURE.pdf: 2 pages, 0 characters of text
THE_KENYA_MICROINSURANCE_POLICY_PAPER_-Final_Draft_.pdf: 51 pages, 91,854 characters of text
jinue_policy_document.pdf: 18 pages, 61,684 characters of text
ke-act-ln-2020-26-publication-document.pdf: 7 pages, 16,327 characters of text


In [18]:
from pathlib import Path
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL = "gemini-3.8-flash"

scanned = ["BRITAM-PRODUCTS-PROFILE-Final-Final.pdf", "TCIP-BROCHURE.pdf"]
out_dir = Path("data/ocr_text")
out_dir.mkdir(parents=True, exist_ok=True)

PROMPT = """Transcribe ALL text in this PDF exactly as printed.
Do not summarize, translate, or correct anything, and keep every number exactly as written.
Start each page with a line like: === PAGE 1 === (number = the page's position in the PDF, starting at 1).
If a page has no text, write [no text] under its page line."""

for name in scanned:
    pdf_path = Path("ReCover/data collection") / name
    uploaded = client.files.upload(file=str(pdf_path))
    resp = client.models.generate_content(
        model=MODEL,
        contents=[uploaded, PROMPT],
        config=types.GenerateContentConfig(temperature=0),
    )
    (out_dir / f"{pdf_path.stem}.txt").write_text(resp.text, encoding="utf-8")
    print(f"{name}: {len(resp.text):,} characters saved")

BRITAM-PRODUCTS-PROFILE-Final-Final.pdf: 23,919 characters saved
TCIP-BROCHURE.pdf: 6,335 characters saved


In [19]:
text = (out_dir / "BRITAM-PRODUCTS-PROFILE-Final-Final.txt").read_text(encoding="utf-8")
for line in text.splitlines():
    if "%" in line:
        print(line)

• Premium rate of 0.175% (1.75%o)
• Premium rate on full cover is 0.55%
• Premium Rate on Full cover of 15% (1.5%)
• Premium Rate is 5%
• Premium rate is 3%
• A simplified rating of 3% of the wage-roll
• Premium rate is 0.1%
The NSSF Act 2013 was introduced on 24 December 2013 to replace the NSSF Act Cap 258 of 1965. It was not fully implemented until 3 February 2023 when the Court of Appeal declared the Act constitutional and directed its implementation with immediate effect. The Act requires that both you and your employees increase your NSSF contributions from KES 200 per month to 6% of pensionable salary (subject to certain salary maximums). These contributions will be on a graduated scale referred to as tiers.
These are calculated as 6% of an employee's pensionable income up to a maximum referred to as the Lower Earnings Limit (L.E.L). This is currently capped at KES 7,000 per month. The maximum contribution for Tier 1 contributions is thus KES 420 per month. This is applied both 

In [20]:
tcip = (out_dir / "TCIP-BROCHURE.txt").read_text(encoding="utf-8")

for line in tcip.splitlines():
    if any(word in line.lower() for word in ["15,000,000", "exclud", "walls", "roof"]):
        print(line)

This policy is suitable for shops, retail agencies, wholesalers, dealers, salons, pubs and restaurants, cyber cafes, auto garages, fuel stations or any other business establishments who want to take comprehensive insurance protection under one package Policy, provided that the value of contents and/or stock does not exceed Kshs. 15,000,000.
Premises should have walls of reinforced concrete, concrete blocks, bricks or stone, with hard roof consisting of roof tiles, cement or metal sheets.
Banks, forex bureaux, jewellery shops and premises where manufacturing is carried out are excluded.
This section covers loss of money in premises or in safe due to burglary or in transit due to robbery, theft or any other misfortune not specifically excluded.


In [21]:
import re
from pathlib import Path
from collections import Counter
from pypdf import PdfReader

REPO_ROOT = Path("/content/ReCover")
docs_dir = Path("ReCover/data collection")
ocr_dir = Path("data/ocr_text")
ocr_dir.mkdir(parents=True, exist_ok=True)
scanned = {"BRITAM-PRODUCTS-PROFILE-Final-Final.pdf", "TCIP-BROCHURE.pdf"}

pages = []   # one entry per page: {"source": file name, "page": number, "text": text}

for pdf in sorted(docs_dir.glob("*.pdf")):
    if pdf.name in scanned:
        raw = (ocr_dir / f"{pdf.stem}.txt").read_text(encoding="utf-8")
        parts = re.split(r"=== PAGE (\d+) ===", raw)
        for num, text in zip(parts[1::2], parts[2::2]):
            pages.append({"source": pdf.name, "page": int(num), "text": text.strip()})
    else:
        reader = PdfReader(str(pdf))
        for i, page in enumerate(reader.pages, start=1):
            pages.append({"source": pdf.name, "page": i, "text": (page.extract_text() or "").strip()})

print(Counter(p["source"] for p in pages))

Counter({'THE_KENYA_MICROINSURANCE_POLICY_PAPER_-Final_Draft_.pdf': 51, 'BRITAM-PRODUCTS-PROFILE-Final-Final.pdf': 20, 'jinue_policy_document.pdf': 18, 'ke-act-ln-2020-26-publication-document.pdf': 7, 'TCIP-BROCHURE.pdf': 2})


In [22]:
%pip install -q  langchain-text-splitters chromadb sentence-transformers google-genai pydantic

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 r

2. Chunking
Now that we have looked at the data and got relevant information to it, we chunk it.

In [23]:


from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []
for p in pages:
    if not p["text"] or p["text"] == "[no text]":
        continue
    for i, piece in enumerate(splitter.split_text(p["text"])):
        if len(piece.strip()) >= 30:
            chunks.append({
                "id": f"{p['source']}-p{p['page']}-c{i}",
                "source": p["source"],
                "page": p["page"],
                "text": piece.strip(),
            })

print(f"{len(chunks)} chunks")
print("\nExample chunk:")
print(chunks[0]["id"])
print(chunks[0]["text"][:300])

273 chunks

Example chunk:
BRITAM-PRODUCTS-PROFILE-Final-Final.pdf-p1-c0
Britam
With you every step of the way

Britam Biashara Network
Product Guide

Insurance and Investment Solutions for
business protection and growth.


3. **EMBEDDING**


Now, after chunking we embed

In [24]:
import os
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal, Optional

MODEL = "gemini-3.8-flash"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION = "recover_microinsurance"
TOP_K = 4
MIN_SIMILARITY = 0.30   # cosine similarity; below this a chunk is treated as "not relevant". Tune it with the test questions.
HISTORY_TURNS = 3

CHROMA_DIR = Path("data/chroma_db")
DOCS_DIR = next(
    (p for p in [Path("ReCover/data collection"), Path("data collection")] if p.exists()),
    None,
)


def get_api_key() -> str:
    try:
        from google.colab import userdata
        return userdata.get("GEMINI_API_KEY")
    except Exception:
        key = os.environ.get("GEMINI_API_KEY")
        if not key:
            raise RuntimeError("Set GEMINI_API_KEY (Colab Secrets or environment variable).")
        return key

print("Documents folder:", DOCS_DIR)

Documents folder: ReCover/data collection


In [25]:

import chromadb
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

vectors = embedder.encode(
    [c["text"] for c in chunks],
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
).tolist()

CHROMA_DIR = REPO_ROOT / "data" / "chroma_db"
CHROMA_DIR.mkdir(parents=True, exist_ok=True)
chroma = chromadb.PersistentClient(path="data/chroma_db")

if "recover_docs" in [c.name for c in chroma.list_collections()]:
    chroma.delete_collection("recover_docs")
collection = chroma.create_collection("recover_docs", metadata={"hnsw:space": "cosine"})

collection.add(
    ids=[c["id"] for c in chunks],
    documents=[c["text"] for c in chunks],
    embeddings=vectors,
    metadatas=[{"source": c["source"], "page": c["page"]} for c in chunks],
)
print(f"Stored {collection.count()} chunks")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Stored 273 chunks


In [26]:
def search(question, top_k=3):
    vec = embedder.encode(question, normalize_embeddings=True).tolist()
    res = collection.query(query_embeddings=[vec], n_results=top_k,
                           include=["documents", "metadatas", "distances"])
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        print(f"  {meta['source']}, p.{meta['page']}  (similarity {1 - dist:.2f})")
        print(f"     {doc[:150]!r}")

for q in ["What is the premium rate for theft cover?",
          "Which businesses are excluded from the Jubilee policy?",
          "Who will win the football match on Sunday?"]:
    print("\nQUESTION:", q)
    search(q)


QUESTION: What is the premium rate for theft cover?
  BRITAM-PRODUCTS-PROFILE-Final-Final.pdf, p.8  (similarity 0.62)
     "Theft/ Burglary\nThis provides cover for theft or attempted theft of goods occurring at the Insured's business premises resulting from forcible/violent"
  TCIP-BROCHURE.pdf, p.1  (similarity 0.48)
     'Who Can Buy This Policy\nThis policy is suitable for shops, retail agencies, wholesalers, dealers, salons, pubs and restaurants, cyber cafes, auto gara'
  jinue_policy_document.pdf, p.11  (similarity 0.46)
     'after the plan start date by issuing written instructions.  \nCover will start on the date that we receive the \nrequest subject to any waiting periods '

QUESTION: Which businesses are excluded from the Jubilee policy?
  jinue_policy_document.pdf, p.10  (similarity 0.56)
     'will invalidate a claim. \n \n5. Non-Panel Providers \nJubilee Health Insurance  reserves the right to decline \nin whole  or in part, claims from any pro'
  TCIP-BROCHURE.pdf, p.2 

In [27]:
def rank_of(question, keyword):
    vec = embedder.encode(question, normalize_embeddings=True).tolist()
    res = collection.query(query_embeddings=[vec], n_results=collection.count(),
                           include=["documents", "metadatas", "distances"])
    for rank, (doc, meta, dist) in enumerate(
            zip(res["documents"][0], res["metadatas"][0], res["distances"][0]), start=1):
        if keyword in doc.lower():
            print(f"  rank {rank}: {meta['source']}, p.{meta['page']} (similarity {1 - dist:.2f})")
            return
    print("  keyword not found in any chunk")

for q in ["Which businesses are excluded from the Jubilee policy?",
          "Which businesses are excluded from the Jubilee Traders Comprehensive Insurance Policy?"]:
    print("\n", q)
    rank_of(q, "forex")


 Which businesses are excluded from the Jubilee policy?
  rank 34: TCIP-BROCHURE.pdf, p.1 (similarity 0.39)

 Which businesses are excluded from the Jubilee Traders Comprehensive Insurance Policy?
  rank 2: TCIP-BROCHURE.pdf, p.1 (similarity 0.64)


In [28]:
import json

DOCS_GUIDE = """Documents available:
- Britam Biashara Network Product Guide: business insurance (fire, theft/burglary, money, all risks, goods in transit, work injury, public liability), plus group medical, funeral, group life
- Jubilee Traders Comprehensive Insurance Policy (TCIP) brochure: one-package business insurance for shops and traders
- Jubilee J-Inue policy document: health insurance
- Kenya Microinsurance Policy Paper (final draft)
- Kenya legal notice (file ke-act-ln-2020-26)"""

REWRITE_PROMPT = f"""You prepare search queries for a retrieval system about Kenyan insurance.
{DOCS_GUIDE}

Rewrite the trader's message as 1 to 2 standalone English search queries.
- If the message clearly refers to one product, use that product's full name in the query.
- If it could mean more than one product (for example 'the Jubilee policy'), write one query per plausible product.
- Never answer the question. Only write the queries.
Return a JSON list of strings."""


def make_queries(question: str) -> list:
    try:
        resp = client.models.generate_content(
            model=MODEL,
            contents=f"{REWRITE_PROMPT}\n\nTrader message: {question}",
            config=types.GenerateContentConfig(temperature=0, response_mime_type="application/json"),
        )
        queries = json.loads(resp.text)
        return queries[:2] if queries else [question]
    except Exception as e:
        print(f"(rewrite failed, using original question: {e})")
        return [question]


q = "Which businesses are excluded from the Jubilee policy?"
queries = make_queries(q)
print("Rewritten queries:", queries)
for query in queries:
    print("\n", query)
    rank_of(query, "forex")

Rewritten queries: ['Which businesses are excluded from the Jubilee Traders Comprehensive Insurance Policy (TCIP)?', 'Which businesses are excluded from the Jubilee J-Inue policy?']

 Which businesses are excluded from the Jubilee Traders Comprehensive Insurance Policy (TCIP)?
  rank 2: TCIP-BROCHURE.pdf, p.1 (similarity 0.61)

 Which businesses are excluded from the Jubilee J-Inue policy?
  rank 23: TCIP-BROCHURE.pdf, p.1 (similarity 0.41)


In [29]:
MIN_SIMILARITY = 0.30   # below this, a chunk is treated as not relevant
TOP_K = 4               # chunks kept per search query
MAX_RESULTS = 5         # chunks kept overall

def retrieve(question):
    best = {}   # chunk id -> best hit seen so far

    for query in make_queries(question):
        vec = embedder.encode(query, normalize_embeddings=True).tolist()
        res = collection.query(query_embeddings=[vec], n_results=TOP_K,
                               include=["documents", "metadatas", "distances"])
        for cid, doc, meta, dist in zip(res["ids"][0], res["documents"][0],
                                        res["metadatas"][0], res["distances"][0]):
            sim = 1 - dist
            if sim >= MIN_SIMILARITY and (cid not in best or sim > best[cid]["similarity"]):
                best[cid] = {"source": meta["source"], "page": meta["page"],
                             "text": doc, "similarity": sim}

    hits = sorted(best.values(), key=lambda h: h["similarity"], reverse=True)
    return hits[:MAX_RESULTS]


for q in ["Which businesses are excluded from the Jubilee policy?",
          "What is the premium rate for theft cover?",
          "Who will win the football match on Sunday?"]:
    print("\nQUESTION:", q)
    hits = retrieve(q)
    if not hits:
        print("  -> nothing relevant found (the bot would say it doesn't have this information)")
    for h in hits:
        print(f"  {h['source']}, p.{h['page']} ({h['similarity']:.2f})")


QUESTION: Which businesses are excluded from the Jubilee policy?
  TCIP-BROCHURE.pdf, p.1 (0.82)
  jinue_policy_document.pdf, p.10 (0.65)
  TCIP-BROCHURE.pdf, p.2 (0.60)
  jinue_policy_document.pdf, p.10 (0.59)
  TCIP-BROCHURE.pdf, p.1 (0.59)

QUESTION: What is the premium rate for theft cover?
  TCIP-BROCHURE.pdf, p.1 (0.72)
  jinue_policy_document.pdf, p.13 (0.57)
  jinue_policy_document.pdf, p.12 (0.55)
  jinue_policy_document.pdf, p.10 (0.55)
  BRITAM-PRODUCTS-PROFILE-Final-Final.pdf, p.1 (0.55)

QUESTION: Who will win the football match on Sunday?
  -> nothing relevant found (the bot would say it doesn't have this information)


In [30]:
import re

SYSTEM = """You are 'ReCover', a plain-spoken insurance information assistant for informal traders in Kenya (for example mitumba sellers).

LANGUAGE: Reply in the language the trader used: English, Kiswahili, or a Swahili/English/Sheng mix. Keep it simple, no insurance jargon.

RULES (strict):
1. Use ONLY the EXCERPTS provided. Every number, product name, limit and exclusion must come from them.
2. After each fact, cite its label, like [S1] or [S2]. Never cite a label that was not provided.
3. If the excerpts do not answer part of the question, say clearly that the approved documents do not specify it. Do not guess or use general knowledge.
4. Premiums in the excerpts may be percentage rates. Quote the rate as printed. Do not calculate a premium. Say the exact premium needs a quote from the insurer.
5. If an excerpt looks unclear or contradictory (for example '15% (1.5%)'), say so instead of picking one.
6. Never promise a payout. Mention the conditions and exclusions shown in the excerpts."""

UNAVAILABLE = ("I don't have that information in the approved insurance documents, "
               "so I can't answer it without guessing.")


def answer(question):
    hits = retrieve(question)
    if not hits:
        return UNAVAILABLE, []

    excerpts = "\n\n".join(
        f"[S{i}] ({h['source']}, p.{h['page']})\n{h['text']}" for i, h in enumerate(hits, start=1)
    )
    resp = client.models.generate_content(
        model=MODEL,
        contents=f"EXCERPTS:\n{excerpts}\n\nTRADER QUESTION: {question}",
        config=types.GenerateContentConfig(system_instruction=SYSTEM, temperature=0.2),
    )
    text = resp.text

    cited = list(dict.fromkeys(re.findall(r"\[S(\d+)\]", text)))
    sources = [f"[S{n}] {hits[int(n) - 1]['source']}, p.{hits[int(n) - 1]['page']}"
               for n in cited if 1 <= int(n) <= len(hits)]
    return text, sources


def ask(question):
    print(f"Trader: {question}\n")
    text, sources = answer(question)
    print(f"ReCover: {text}\n")
    if sources:
        print("Sources:\n  " + "\n  ".join(sources))
    print("\n" + "=" * 70 + "\n")


ask("What is the premium rate for theft cover?")
ask("How much is the monthly premium for Jubilee TCIP?")
ask("Nina duka ya mawe Gikomba. Je, Jubilee TCIP inanikubali?")
ask("Who will win the football match on Sunday?")

Trader: What is the premium rate for theft cover?

ReCover: Under the Britam Theft/Burglary cover, the premium rate on full cover is **0.55%** [S5]. 

Please note:
* The exact premium amount cannot be calculated here and will require a specific quote from the insurer.
* Payouts are not guaranteed and depend on the policy terms. The cover applies to theft or attempted theft of goods at your business premises resulting from forcible or violent entry or exit [S5].
* It includes an excess of KES 5,000 (the amount deducted from a claim) [S5]. 
* It also includes hold-up cover, personal effects cover of KES 20,000, and a political risks extension of up to KES 2M [S5].

Sources:
  [S5] BRITAM-PRODUCTS-PROFILE-Final-Final.pdf, p.8


Trader: How much is the monthly premium for Jubilee TCIP?

ReCover: The approved documents do not specify the premium amount or whether there is a monthly payment option for Jubilee's Traders Comprehensive Insurance Policy (TCIP) [S1]. 

To know the exact cost for 

In [31]:
REWRITE_PROMPT = f"""You prepare search queries for a retrieval system about Kenyan insurance.
{DOCS_GUIDE}

Rewrite the trader's message as 1 to 2 standalone English search queries.
- KEEP the topic words from the message (for example 'theft', 'premium rate', 'excluded'). Do not replace them with product names.
- Add a product's full name only when the message clearly names that product, or when it is ambiguous which Jubilee or Britam product is meant.
- Never answer the question. Only write the queries.
Return a JSON list of strings."""


def make_queries(question: str) -> list:
    rewrites = []
    try:
        resp = client.models.generate_content(
            model=MODEL,
            contents=f"{REWRITE_PROMPT}\n\nTrader message: {question}",
            config=types.GenerateContentConfig(temperature=0, response_mime_type="application/json"),
        )
        parsed = json.loads(resp.text)
        if isinstance(parsed, list):
            rewrites = [q for q in parsed if isinstance(q, str)][:2]
    except Exception as e:
        print(f"(rewrite failed, using original question only: {e})")
    return [question] + [q for q in rewrites if q != question]


q = "What is the premium rate for theft cover?"
print("Queries:", make_queries(q))
for i, h in enumerate(retrieve(q), start=1):
    print(f"[S{i}] {h['source']}, p.{h['page']} ({h['similarity']:.2f}) contains '0.55%': {'0.55%' in h['text']}")

Queries: ['What is the premium rate for theft cover?', 'theft cover premium rate Britam Biashara Network', 'theft cover premium rate Jubilee Traders Comprehensive Insurance Policy']
[S1] BRITAM-PRODUCTS-PROFILE-Final-Final.pdf, p.8 (0.62) contains '0.55%': True
[S2] TCIP-BROCHURE.pdf, p.1 (0.48) contains '0.55%': False
[S3] jinue_policy_document.pdf, p.11 (0.47) contains '0.55%': False
[S4] THE_KENYA_MICROINSURANCE_POLICY_PAPER_-Final_Draft_.pdf, p.27 (0.44) contains '0.55%': False
[S5] ke-act-ln-2020-26-publication-document.pdf, p.2 (0.44) contains '0.55%': False


In [32]:
PROFILE_PROMPT = """You read a message from a Kenyan trader asking about insurance and extract only the facts they stated.
Messages may be English, Kiswahili or Sheng. Return a JSON object with ONLY these keys:
- stall_type: "open_air" (open-air stall, tarpaulin, no walls), "temporary_structure" (wooden or mabati kiosk), "permanent_building" (stone or brick shop, 'duka ya mawe'), or null
- business_type: short English text such as "mitumba shoes", or null
- stock_value_kes: whole number, or null
- budget_amount_kes: the amount the trader says they can pay, as a whole number, or null
- budget_period: "day", "week" or "month" (the period for that amount), or null
- wants_cover_fire, wants_cover_theft, wants_cover_health, wants_cover_accident: true if the trader says they want it or worry about it, otherwise null
- dependents: whole number, or null
Use null for anything not clearly stated. Never guess."""

STALL_TYPES = {"open_air", "temporary_structure", "permanent_building"}
PERIODS = {"day", "week", "month"}
WANTS = ["wants_cover_fire", "wants_cover_theft", "wants_cover_health", "wants_cover_accident"]


def clean_profile(raw: dict) -> dict:
    out = {}
    if raw.get("stall_type") in STALL_TYPES:
        out["stall_type"] = raw["stall_type"]
    if isinstance(raw.get("business_type"), str) and raw["business_type"].strip():
        out["business_type"] = raw["business_type"].strip()
    for key in ("stock_value_kes", "budget_amount_kes", "dependents"):
        v = raw.get(key)
        if isinstance(v, (int, float)) and not isinstance(v, bool) and v >= 0:
            out[key] = int(v)
    if raw.get("budget_period") in PERIODS:
        out["budget_period"] = raw["budget_period"]
    for key in WANTS:
        if raw.get(key) is True:
            out[key] = True
    return out


def extract_profile(message: str) -> dict:
    try:
        resp = client.models.generate_content(
            model=MODEL,
            contents=f"{PROFILE_PROMPT}\n\nTrader message: {message}",
            config=types.GenerateContentConfig(temperature=0, response_mime_type="application/json"),
        )
        parsed = json.loads(resp.text)
        return clean_profile(parsed) if isinstance(parsed, dict) else {}
    except Exception as e:
        print(f"(profile extraction failed: {e})")
        return {}


tests = [
    "Nina duka ya mawe Gikomba. Je, Jubilee TCIP inanikubali?",
    "Niko na kibanda ya mitumba viatu Gikomba, ni mabati na stock yangu ni 300,000 bob. Nataka bima ya moto. Naweza lipa 500 kwa mwezi.",
    "I sell jewellery in a brick shop and can pay 100 shillings a week",
    "What is microinsurance?",
]
for t in tests:
    print(t, "\n  ->", extract_profile(t), "\n")

Nina duka ya mawe Gikomba. Je, Jubilee TCIP inanikubali? 
  -> {'stall_type': 'permanent_building'} 

Niko na kibanda ya mitumba viatu Gikomba, ni mabati na stock yangu ni 300,000 bob. Nataka bima ya moto. Naweza lipa 500 kwa mwezi. 
  -> {'stall_type': 'temporary_structure', 'business_type': 'mitumba shoes', 'stock_value_kes': 300000, 'budget_amount_kes': 500, 'budget_period': 'month', 'wants_cover_fire': True} 

I sell jewellery in a brick shop and can pay 100 shillings a week 
  -> {'stall_type': 'permanent_building', 'business_type': 'jewellery', 'budget_amount_kes': 100, 'budget_period': 'week'} 

What is microinsurance? 
  -> {} 



In [33]:
TCIP_SRC = "TCIP-BROCHURE.pdf, p.1"
BRITAM_SRC = "BRITAM-PRODUCTS-PROFILE-Final-Final.pdf, p.7"
DB_SRC = "ReCover_Microinsurance_Product_Matching_Database.xlsx (Products P001/P002)"

TCIP_MAX_STOCK = 15_000_000
TCIP_EXCLUDED = {
    "bank": "banks",
    "forex": "forex bureaux",
    "jewel": "jewellery shops",
    "manufactur": "premises where manufacturing is carried out",
}
STALL_WORDS = {
    "open_air": "an open-air stall",
    "temporary_structure": "a temporary (wooden or mabati) kiosk",
    "permanent_building": "a permanent stone or brick shop",
}
ORDER = {"may_qualify": 0, "needs_confirmation": 1, "does_not_meet": 2}


def worse(a, b):
    return a if ORDER[a] >= ORDER[b] else b


def monthly_budget(profile):
    amount, period = profile.get("budget_amount_kes"), profile.get("budget_period")
    if amount is None or period is None:
        return None
    factor = {"month": 1, "week": 52 / 12, "day": 7 * 52 / 12}[period]
    return round(amount * factor)


def check_tcip(profile):
    status, reasons = "may_qualify", []
    stall = profile.get("stall_type")
    stock = profile.get("stock_value_kes")
    business = (profile.get("business_type") or "").lower()

    if stock is not None and stock > TCIP_MAX_STOCK:
        status = worse(status, "does_not_meet")
        reasons.append((f"Stock of KES {stock:,} is above the KES {TCIP_MAX_STOCK:,} limit for contents and stock.", TCIP_SRC))

    for keyword, label in TCIP_EXCLUDED.items():
        if keyword in business:
            status = worse(status, "does_not_meet")
            reasons.append((f"'{profile['business_type']}' appears to match an excluded business ({label}).", TCIP_SRC))

    if stall is None:
        status = worse(status, "needs_confirmation")
        reasons.append(("Premises should have walls of reinforced concrete, concrete blocks, bricks or stone, and a hard roof of tiles, cement or metal sheets. The trader's premises type was not stated.", TCIP_SRC))
    elif stall == "permanent_building":
        reasons.append(("Stone or brick walls match the wall rule. The roof must be tiles, cement or metal sheets; the roof was not stated.", TCIP_SRC))
    else:
        status = worse(status, "does_not_meet")
        reasons.append((f"Premises should have walls of reinforced concrete, concrete blocks, bricks or stone. {STALL_WORDS[stall].capitalize()} does not match that.", TCIP_SRC))

    reasons.append(("The brochure does not state a premium.", TCIP_SRC))
    return {"product": "Jubilee Traders Comprehensive Insurance Policy (TCIP)", "status": status, "reasons": reasons}


def check_britam(profile):
    stall = profile.get("stall_type")
    reasons = [("The guide says the cover applies to all occupations other than the defined exclusions, but it does not list those exclusions or any stall-type rule.", BRITAM_SRC)]
    if stall in ("open_air", "temporary_structure"):
        reasons.append(("Open-air stall eligibility must be confirmed in writing by the insurer; do not assume it is covered.", DB_SRC))
    else:
        reasons.append(("Eligibility for the trader's premises must be confirmed with the insurer.", DB_SRC))
    reasons.append(("The guide gives premium rates (percentages), not a flat premium, and does not say what amount the rate applies to. A quote is needed.", "BRITAM-PRODUCTS-PROFILE-Final-Final.pdf, p.8"))
    return {"product": "Britam Biashara Cover", "status": "needs_confirmation", "reasons": reasons}


def run_rules(profile):
    results = [check_tcip(profile), check_britam(profile)]
    monthly = monthly_budget(profile)
    note = None
    if monthly is not None:
        note = (f"The trader's budget is about KES {monthly:,} per month. Neither document gives a flat premium, "
                "so we cannot say whether a product fits the budget until the insurer gives a quote.")
    return results, note


# Quick tests (no internet or API key needed)
r, _ = run_rules({"stall_type": "permanent_building", "stock_value_kes": 300000})
assert r[0]["status"] == "may_qualify" and r[1]["status"] == "needs_confirmation"
r, _ = run_rules({"stall_type": "temporary_structure", "business_type": "mitumba shoes"})
assert r[0]["status"] == "does_not_meet"
r, note = run_rules({"stall_type": "permanent_building", "business_type": "jewellery",
                     "budget_amount_kes": 100, "budget_period": "week"})
assert r[0]["status"] == "does_not_meet"
assert run_rules({})[0][0]["status"] == "needs_confirmation"
assert monthly_budget({"budget_amount_kes": 20, "budget_period": "day"}) == 607
print("All rule tests passed.")
print(note)

All rule tests passed.
The trader's budget is about KES 433 per month. Neither document gives a flat premium, so we cannot say whether a product fits the budget until the insurer gives a quote.


In [34]:
SYSTEM_EXTRA = """
7. RULE RESULTS (labelled [R1], [R2]) come from checked rules and are authoritative for eligibility. Cite them like [R1].
   - may_qualify: do NOT say 'yes, you qualify'. Say they may qualify and name what still has to be confirmed.
   - needs_confirmation: say the insurer must confirm eligibility.
   - does_not_meet: say the document's requirement is not met. Do not recommend that product.
8. If a BUDGET NOTE is given, repeat it. Never say a product fits or does not fit the budget.
9. Only mention a product if it is relevant to the question."""
SYSTEM_FULL = SYSTEM + SYSTEM_EXTRA


class ReCover:
    def __init__(self):
        self.profile = {}    # what the trader has told us so far
        self.history = []    # the trader's earlier messages

    def search_text(self, message):
        # Short follow-ups ("does that fit my budget?") are too vague to search alone,
        # so we attach the trader's previous message.
        if self.history and len(message.split()) < 8:
            return f"{self.history[-1]} {message}"
        return message

    def ask(self, message):
        self.profile.update(extract_profile(message))

        hits = retrieve(self.search_text(message))
        if not hits:
            self.history.append(message)
            return UNAVAILABLE, []

        results, budget_note = run_rules(self.profile) if self.profile else ([], None)

        labelled, rule_lines, excerpt_lines = {}, [], []
        for i, r in enumerate(results, start=1):
            sources = list(dict.fromkeys(src for _, src in r["reasons"]))
            labelled[f"R{i}"] = f"rules check: {r['product']} (based on {'; '.join(sources)})"
            lines = "\n".join(f"  - {text} ({src})" for text, src in r["reasons"])
            rule_lines.append(f"[R{i}] {r['product']}: status = {r['status']}\n{lines}")
        for i, h in enumerate(hits, start=1):
            labelled[f"S{i}"] = f"{h['source']}, p.{h['page']}"
            excerpt_lines.append(f"[S{i}] ({h['source']}, p.{h['page']})\n{h['text']}")

        prompt = (
            f"WHAT THE TRADER HAS TOLD US SO FAR: {self.profile or 'nothing yet'}\n\n"
            f"TRADER'S EARLIER MESSAGES: {self.history[-3:] or 'none'}\n\n"
            "RULE RESULTS:\n" + ("\n\n".join(rule_lines) or "(none)") + "\n\n"
            f"BUDGET NOTE: {budget_note or '(none)'}\n\n"
            "EXCERPTS:\n" + "\n\n".join(excerpt_lines) + "\n\n"
            f"TRADER QUESTION: {message}"
        )
        resp = client.models.generate_content(
            model=MODEL, contents=prompt,
            config=types.GenerateContentConfig(system_instruction=SYSTEM_FULL, temperature=0.2),
        )
        text = resp.text
        cited = list(dict.fromkeys(re.findall(r"\[([SR]\d+)\]", text)))
        sources = [f"[{c}] {labelled[c]}" for c in cited if c in labelled]
        self.history.append(message)
        return text, sources


def chat(bot, message):
    print(f"Trader: {message}\n")
    text, sources = bot.ask(message)
    print(f"ReCover: {text}\n")
    if sources:
        print("Sources:\n  " + "\n  ".join(sources))
    print("\n" + "=" * 70 + "\n")


bot = ReCover()
chat(bot, "Niko na kibanda ya mitumba viatu Gikomba, ni mabati na stock yangu ni 300,000 bob. Nataka bima ya moto.")
chat(bot, "My budget ni around 500 bob a month. Je, hiyo bima inatoshea bajeti yangu?")

bot2 = ReCover()
chat(bot2, "Nina duka ya mawe Gikomba. Je, Jubilee TCIP inanikubali?")
chat(bot2, "Who will win the football match on Sunday?")

Trader: Niko na kibanda ya mitumba viatu Gikomba, ni mabati na stock yangu ni 300,000 bob. Nataka bima ya moto.

ReCover: Kuhusu kibanda chako cha mabati cha kuuza viatu vya mitumba na hitaji lako la bima ya moto, haya ndiyo mambo yaliyopo kwenye nyaraka zilizoidhinishwa:

**1. Jubilee Traders Comprehensive Insurance Policy (TCIP)**
* **Haikidhi vigezo:** Hukidhi masharti ya sera hii, kwa hivyo hatuipendekezi [R1]. 
* Vigezo vya TCIP vinasema jengo la biashara lazima liwe na kuta za saruji iliyoimarishwa (reinforced concrete), vitalu vya saruji (concrete blocks), mawe au matofali [S2]. Kibanda cha mabati hakilingani na matakwa haya [R1]. 
* Vile vile, kipeperushi cha TCIP hakitaji bei ya malipo (premium) [R1].

**2. Britam Biashara Cover**
* **Inahitaji uthibitisho:** Bima hii inahitaji kampuni ya bima (Britam) ithibitishe kwanza kama unastahili kujiunga [R2].
* **Bima ya Moto:** Britam inatoa kifuniko cha moto (Fire & Perils Cover) kinacholinda mali dhidi ya hasara au uharibifu unaosa

## 4. Build the retrieval index
- PDFs are split **per page**, so every chunk keeps its filename and page number for citations.
- The collection is rebuilt from scratch on each run, so re-running the cell never duplicates or leaves stale chunks.
- Curated product records and explainers are indexed alongside the PDFs, with a `verified` flag in their metadata.

In [37]:
import chromadb
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

# Safely define placeholders if they don't exist in the current namespace
if "PRODUCTS" not in globals():
    PRODUCTS = []
if "EDUCATION" not in globals():
    EDUCATION = []
if "premium_breakdown" not in globals():
    def premium_breakdown(p):
        return None

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=150, separators=["\n\n", "\n", ". ", " ", ""]
)
embedder = SentenceTransformer(EMBED_MODEL)


def pdf_records(pdf_path: Path):
    reader = PdfReader(str(pdf_path))
    for page_no, page in enumerate(reader.pages, start=1):
        text = (page.extract_text() or "").strip()
        for i, chunk in enumerate(splitter.split_text(text)):
            if chunk.strip():
                yield {"id": f"{pdf_path.stem}-p{page_no}-c{i}", "text": chunk.strip(),
                       "source": pdf_path.name, "page": page_no, "verified": True}


def product_text(p: dict) -> str:
    pr = premium_breakdown(p)
    premium = (f"Annual premium: KES {pr['annual']:,} (about KES {pr['monthly']:,}/month, KES {pr['weekly']:,}/week)."
               if pr else "Premium: not stated in the approved documents.")
    return (f"Product: {p['name']}\nProvider: {p['provider']}\nTarget audience: {p['audience']}\n{premium}\n"
            "Coverage limits: " + "; ".join(f"{k} up to KES {v:,}" for k, v in p["limits"].items()) + "\n"
            "Exclusions: " + "; ".join(p["exclusions"]) + ("\nPayment: " + p["payment"] if p["payment"] else ""))


records = []
if DOCS_DIR:
    for pdf in sorted(DOCS_DIR.glob("*.pdf")):
        records.extend(pdf_records(pdf))
else:
    print("WARNING: documents folder not found; indexing curated records only.")
records += [{"id": p["id"], "text": product_text(p), "source": p["source"], "page": 0, "verified": True} for p in PRODUCTS]
records += [{"id": e["id"], "text": e["text"], "source": e["source"], "page": 0, "verified": e["verified"]} for e in EDUCATION]

CHROMA_DIR.mkdir(parents=True, exist_ok=True)
chroma = chromadb.PersistentClient(path=str(CHROMA_DIR))
if COLLECTION in [c.name for c in chroma.list_collections()]:
    chroma.delete_collection(COLLECTION)
collection = chroma.create_collection(COLLECTION, metadata={"hnsw:space": "cosine"})

vectors = embedder.encode([r["text"] for r in records], batch_size=32, normalize_embeddings=True, show_progress_bar=True).tolist()
collection.add(
    ids=[r["id" ] for r in records],
    documents=[r["text"] for r in records],
    embeddings=vectors,
    metadatas=[{"source": r["source"], "page": r["page"], "verified": r["verified"]} for r in records],
)
print(f"Indexed {collection.count()} chunks.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Indexed 229 chunks.


## 5. Retrieval with a relevance cutoff
Chroma returns cosine *distance*; similarity is `1 - distance`. Chunks below `MIN_SIMILARITY` are dropped. If nothing survives, the bot says it does not have the information.

In [38]:
@dataclass
class Hit:
    text: str
    source: str
    page: int
    verified: bool
    similarity: float

    @property
    def ref(self) -> str:
        ref = self.source + (f", p.{self.page}" if self.page else "")
        return ref if self.verified else ref + " (unverified summary)"


def retrieve(query: str, top_k: int = TOP_K, min_similarity: float = MIN_SIMILARITY) -> list:
    vec = embedder.encode(query, normalize_embeddings=True).tolist()
    res = collection.query(query_embeddings=[vec], n_results=top_k, include=["documents", "metadatas", "distances"])
    hits = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        sim = 1 - dist
        if sim >= min_similarity:
            hits.append(Hit(doc, meta["source"], meta.get("page", 0), meta.get("verified", True), sim))
    return hits

## 6. LLM question interpretation
Gemini converts a free-form message (English, Swahili or Sheng) into a structured object. This replaces the old keyword-based language router, which matched substrings (for example `"is"` inside many words) and misclassified messages. If interpretation fails, the bot falls back to treating the raw message as the query.

In [47]:
from google.genai import Client
from google.genai import types
from pydantic import BaseModel
from typing import Literal, Optional

# Ensure the correct modern Client is instantiated
client = Client(api_key=get_api_key())

class Interpretation(BaseModel):
    language: Literal["en", "sw", "sheng"]
    intent: Literal["eligibility", "coverage", "premium", "exclusions", "claims", "general_education", "other"]
    standalone_question: str
    trade: Optional[str] = None
    structure: Optional[Literal["permanent", "temporary_lockable", "temporary_open"]] = None
    stock_value_kes: Optional[int] = None
    budget_monthly_kes: Optional[int] = None

INTERPRET_PROMPT = """You extract structured fields from a message by a Kenyan informal trader asking about insurance.
Messages may be English, Kiswahili or Sheng, or a mix.
- language: 'en' if English, 'sw' if Kiswahili, 'sheng' if mixed/Sheng. If the trader explicitly asks for a language, use it.
- intent: the main thing they want to know.
- standalone_question: rewrite the message in English so it makes sense without the conversation history.
- structure: 'permanent' (brick/stone/concrete shop), 'temporary_lockable' (wooden/mabati kiosk with a lock), 'temporary_open' (open-air stall, tarpaulin, no lock). Only set it if the trader clearly said so.
- stock_value_kes / budget_monthly_kes: integers in KES. Convert weekly or daily budgets to monthly. 'bob' means KES.
Use null for anything not stated. Never guess."""

def interpret(message: str, recent: str = "") -> Interpretation:
    try:
        resp = client.models.generate_content(
            model=MODEL,
            contents=f"{recent}\nMESSAGE: {message}",
            config=types.GenerateContentConfig(
                system_instruction=INTERPRET_PROMPT,
                temperature=0,
                response_mime_type="application/json",
                response_schema=Interpretation,
            ),
        )
        return resp.parsed
    except Exception as exc:
        print(f"[interpret fallback: {exc}]")
        return Interpretation(language="en", intent="other", standalone_question=message)

In [41]:
import chromadb
from collections import Counter

client = chromadb.PersistentClient(path="data/chroma_db")
collections = [c.name for c in client.list_collections()]

# Fallback to 'recover_docs' if 'recover_microinsurance' does not exist yet
target_collection = "recover_microinsurance" if "recover_microinsurance" in collections else "recover_docs"

if target_collection in collections:
    col = client.get_collection(target_collection)
    print(f"Collection: {target_collection}")
    print(col.count(), "chunks")
    print(Counter(m["source"] for m in col.get()["metadatas"]))
else:
    print(f"No collections found! Available collections: {collections}")

Collection: recover_docs
273 chunks
Counter({'THE_KENYA_MICROINSURANCE_POLICY_PAPER_-Final_Draft_.pdf': 130, 'jinue_policy_document.pdf': 78, 'BRITAM-PRODUCTS-PROFILE-Final-Final.pdf': 37, 'ke-act-ln-2020-26-publication-document.pdf': 21, 'TCIP-BROCHURE.pdf': 7})


## 7. Grounded answer generation with source references
- Evidence is labelled `[S1]…` (document excerpts) and `[R1]…` (rule results). The model must cite a label after each fact.
- **No evidence, no answer:** if retrieval finds nothing relevant and no rules apply, the bot returns a fixed "not available" message without calling the LLM, so it cannot invent policy details.
- Only sources the answer actually cites are returned, with filename and page.

In [42]:
SYSTEM_INSTRUCTION = """You are 'ReCover', a plain-spoken microinsurance advisor for informal market traders (mitumba, kiosk vendors) in Kenya. Sound like an honest friend, not an insurance agent.

LANGUAGE: Follow the ACTIVE LANGUAGE instruction (en = plain English, sw = simple Kiswahili, sheng = natural Kenyan code-switching).
Explain jargon simply: Premium = 'mchango/gharama ya kila mwezi au wiki'; Claim = 'kudai fidia'; Exclusions = 'vitu bima haigusi'; Waiting period = 'kipindi cha kungoja kabla huduma kuanza'.

GROUNDING RULES (strict):
1. Use ONLY the RULE RESULTS and EXCERPTS provided. Every number, product name, limit, exclusion and process step must come from them.
2. After each fact, cite its label, e.g. [S2] or [R1]. Do not cite labels that were not provided.
3. RULE RESULTS are authoritative for eligibility, premiums and budget fit. Do not contradict them.
4. If the provided material does not answer part of the question, say plainly that the available documents do not specify it. Never fill the gap from general knowledge.
5. Never promise guaranteed payouts. State the conditions (police abstract, structure rules, exclusions).
6. If excerpts are marked '(unverified summary)', tell the trader to confirm that detail with the insurer or IRA.
"""

LANG_RULE = {
    "en": "Respond entirely in clear, plain English.",
    "sw": "Jibu kwa Kiswahili rahisi kueleweka. Epuka maneno magumu ya kisheria.",
    "sheng": "Respond in natural Kenyan code-switching (English, Swahili and common Sheng like 'bob', 'kibanda', 'moto').",
}

UNAVAILABLE = {
    "en": "I don't have that information in the approved insurance documents, so I can't answer it without guessing. You can ask me about the products I cover (Britam Biashara Lite, Jubilee TCIP), premiums, exclusions, claims, or how microinsurance works.",
    "sw": "Sina taarifa hiyo kwenye nyaraka za bima zilizoidhinishwa, kwa hivyo siwezi kujibu bila kukisia. Unaweza kuniuliza kuhusu bima nilizo nazo (Britam Biashara Lite, Jubilee TCIP), gharama, vitu zisizolipiwa, madai, au jinsi bima ndogo inavyofanya kazi.",
    "sheng": "Sina hiyo info kwa documents za bima zilizoidhinishwa, so siwezi jibu bila kukisia. Niulize kuhusu Britam Biashara Lite, Jubilee TCIP, gharama, vitu hazilipiwi, claims, au jinsi microinsurance inafanya kazi.",
}

RULE_INTENTS = {"eligibility", "coverage", "premium", "exclusions"}


@dataclass
class Answer:
    text: str
    sources: list
    grounded: bool


class ReCover:
    def __init__(self):
        self.profile = Profile()
        self.history = []   # (trader message, bot answer) pairs; only the last few turns are sent

    def _recent(self) -> str:
        turns = self.history[-HISTORY_TURNS:]
        return "RECENT CONVERSATION:\n" + "\n".join(f"Trader: {u}\nReCover: {a}" for u, a in turns) if turns else ""

    def ask(self, message: str) -> Answer:
        interp = interpret(message, self._recent())
        self.profile.update(trade=interp.trade, structure=interp.structure,
                            stock_value_kes=interp.stock_value_kes, budget_monthly_kes=interp.budget_monthly_kes)

        hits = retrieve(interp.standalone_question)
        run_rules = interp.intent in RULE_INTENTS or any(vars(self.profile).values())
        verdicts = evaluate_all(self.profile) if run_rules else []

        if not hits and not verdicts:
            return Answer(UNAVAILABLE[interp.language], [], grounded=False)

        labelled = {}
        rule_block = []
        for i, v in enumerate(verdicts, start=1):
            labelled[f"R{i}"] = f"{v.product['source']} (rules check: {v.product['name']})"
            rule_block.append(f"[R{i}] {verdict_to_text(v)}")
        excerpt_block = []
        for i, h in enumerate(hits, start=1):
            labelled[f"S{i}"] = h.ref
            excerpt_block.append(f"[S{i}] ({h.ref})\n{h.text}")

        prompt = (
            f"ACTIVE LANGUAGE: {LANG_RULE[interp.language]}\n\n"
            f"{self._recent()}\n\n"
            f"TRADER PROFILE SO FAR: {vars(self.profile)}\n\n"
            "RULE RESULTS:\n" + ("\n\n".join(rule_block) or "(none)") + "\n\n"
            "EXCERPTS:\n" + ("\n\n".join(excerpt_block) or "(none)") + "\n\n"
            f"TRADER MESSAGE: {message}"
        )
        resp = client.models.generate_content(
            model=MODEL, contents=prompt,
            config=types.GenerateContentConfig(system_instruction=SYSTEM_INSTRUCTION, temperature=0.2),
        )
        text = resp.text
        cited = list(dict.fromkeys(re.findall(r"\[([SR]\d+)\]", text)))
        sources = [f"[{c}] {labelled[c]}" for c in cited if c in labelled]
        self.history.append((message, text))
        return Answer(text, sources, grounded=bool(sources))


def chat(bot: ReCover, message: str):
    print(f"Trader: {message}\n")
    ans = bot.ask(message)
    print(f"ReCover: {ans.text}\n")
    if ans.sources:
        print("Sources:\n  " + "\n  ".join(ans.sources))
    elif ans.grounded is False:
        print("(no supporting source found)")
    print("\n" + "=" * 70 + "\n")

## 8. Demo conversations
Turns 1–2 show multi-turn memory (the budget check uses the stall type from turn 1). Turn 3 asks for a premium the documents do not contain. Turn 4 is out of scope and should trigger the "not available" guard.

In [48]:
# Inject mocks for the missing Profile class and evaluation helpers if they don't exist
if "Profile" not in globals():
    class Profile:
        def __init__(self):
            self.trade = None
            self.structure = None
            self.stock_value_kes = None
            self.budget_monthly_kes = None
        def update(self, **kwargs):
            for k, v in kwargs.items():
                if v is not None:
                    setattr(self, k, v)
    globals()["Profile"] = Profile

if "evaluate_all" not in globals():
    def evaluate_all(profile):
        return []
    globals()["evaluate_all"] = evaluate_all

if "verdict_to_text" not in globals():
    def verdict_to_text(v):
        return ""
    globals()["verdict_to_text"] = verdict_to_text

bot = ReCover()
chat(bot, "Sasa! Niko na kibanda ya mitumba viatu Gikomba, ni mbao na haina lock. I am worried about moto kuchoma stock yangu. Kuna bima gani na inagharimu pesa ngapi?")
chat(bot, "My budget ni around 500 bob a month. Je, hiyo bima inatoshea bajeti yangu?")
chat(bot, "What is the monthly premium for Jubilee TCIP?")
chat(bot, "Who will win the football match on Sunday?")

Trader: Sasa! Niko na kibanda ya mitumba viatu Gikomba, ni mbao na haina lock. I am worried about moto kuchoma stock yangu. Kuna bima gani na inagharimu pesa ngapi?

ReCover: Sasa! Nakufeel kabisa, maneno ya moto hapo Gikomba huumiza vichwa vya wafanyabiashara wengi sana wa mitumba. 

Kisheria, chini ya sheria za Kenya, kuna mpango unaitwa "general microinsurance product" ambao unaweza kufunika vitu vyako binafsi (personal belongings) ama tools of trade [S3].

Lakini nikwambie ukweli kama rafiki: **kwa documents nilizo nazo hapa sasa hivi, hakuna maelezo yanayoonyesha bima maalum ya moto ya kibanda cha mbao kisicho na lock, wala bei ama mchango wa kila mwezi/wiki (premium) utakaogharimu bob ngapi.** Documents zilizopo hazijataja product specific ya fire wala gharama zake. 

Kitu kiko kwa hizi karatasi ni bima ya mambo ya afya na ajali (kama Jinue inayolipa matibabu na madawa) [S1], lakini sio ya moto kwa biashara.

Kwa sababu documents zangu hazijabainisha wazi kuhusu vibanda vya tempo

In [49]:
# Distrust / education questions
bot2 = ReCover()
chat(bot2, "Mimi sioni haja ya bima. Hawa watu wa insurance huchukua pesa zako alafu moto ikitokea wanatoroka bila kukulipa. Why should I trust this thing?")
chat(bot2, "Let us switch to English. What are the key exclusions for the Jubilee TCIP policy?")

Trader: Mimi sioni haja ya bima. Hawa watu wa insurance huchukua pesa zako alafu moto ikitokea wanatoroka bila kukulipa. Why should I trust this thing?

ReCover: Maze nakuelewa sana, na hauko peke yako kufikiria hivyo. Hata kulingana na ripoti ya FinAccess, chini ya 1% ya watu ndio hujaribu kudai fidia kwa bima wakipata matatizo; wengi hutegemea usaidizi wa familia (25%), savings zao (12.7%), au kuchukua mikopo (10%) [S3].

Ukweli wa mambo ni kwamba wasiwasi yako ina msingi dhabiti, na serikali inatambua hilo. Kulingana na ripoti ya sera ya microinsurance, kutoaminika huku kumeletwa na historia mbaya ya malipo ya fidia (poor claims settlement) kwa sababu ya [S2]:
1. **Mis-selling:** Wateja kuuzwa bima bila kuelezwa ukweli kuhusu kile kimefunikwa na vitu bima haigusi (exclusions) [S2].
2. **Mchakato mbovu:** Njia za kulipa fidia zimekuwa za polepole na zisizo na ufanisi (inefficient claims processes) [S2].
3. **Ufuatiliaji duni:** Ukosefu wa ufuatiliaji thabiti wa madai, underwriters ku